# Data Preprocessing for xAPI Pass/Fail Prediction

This notebook preprocesses the xAPI-Edu-Data dataset for pass/fail prediction by:
1. Loading the xAPI dataset
2. Creating pass/fail target variable
3. Engineering features (engagement score, absences, parental support, academic level)
4. Encoding categorical variables
5. Normalizing numeric features
6. Preparing the final dataset for modeling

## Target Variable
**Pass/Fail**: Binary classification (1 = Pass, 0 = Fail)

Based on `Class` = "M" or "H" (Medium/High) = Pass, "L" (Low) = Fail


## 1. Setup and Imports


In [1]:
# Install required packages
!pip install -q pandas numpy scikit-learn



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
import os
import warnings
warnings.filterwarnings('ignore')

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully!")


Libraries imported successfully!


## 2. Data Loading


In [3]:
# Find dataset path
# Try multiple possible paths relative to notebook location
possible_paths = [
    '../../datasets/xapi-edu-data/xAPI-Edu-Data.csv',       # From final_code/preprocessing/
    '../../../datasets/xapi-edu-data/xAPI-Edu-Data.csv',    # From final_code/
    'datasets/xapi-edu-data/xAPI-Edu-Data.csv',             # From project root
    '../../../../datasets/xapi-edu-data/xAPI-Edu-Data.csv'  # From pretests/charaka/analysis/pass-fail-prediction/
]

xapi_path = None
for path in possible_paths:
    if os.path.exists(path):
        xapi_path = path
        break

if xapi_path is None:
    raise FileNotFoundError(f"Could not find xAPI dataset. Tried: {possible_paths}")

print(f"xAPI dataset path: {os.path.abspath(xapi_path)}")


xAPI dataset path: /Users/charaka/Desktop/Projects/uom-student-performance-analytics/datasets/xapi-edu-data/xAPI-Edu-Data.csv


In [4]:
# Load xAPI dataset
df_xapi = pd.read_csv(xapi_path)

# Fix xAPI column name capitalization issues
df_xapi.rename(columns={'VisITedResources': 'VisitedResources', 'NationalITy': 'Nationality'}, inplace=True)

print(f"xAPI dataset: {df_xapi.shape[0]} rows × {df_xapi.shape[1]} columns")
print(f"\nColumns: {list(df_xapi.columns)}")
print(f"\nFirst few rows:")
df_xapi.head()


xAPI dataset: 480 rows × 17 columns

Columns: ['gender', 'Nationality', 'PlaceofBirth', 'StageID', 'GradeID', 'SectionID', 'Topic', 'Semester', 'Relation', 'raisedhands', 'VisitedResources', 'AnnouncementsView', 'Discussion', 'ParentAnsweringSurvey', 'ParentschoolSatisfaction', 'StudentAbsenceDays', 'Class']

First few rows:


,gender,Nationality,PlaceofBirth,StageID,GradeID,SectionID,Topic,Semester,Relation,raisedhands,VisitedResources,AnnouncementsView,Discussion,ParentAnsweringSurvey,ParentschoolSatisfaction,StudentAbsenceDays,Class
0,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,15,16,2,20,Yes,Good,Under-7,M
1,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,20,20,3,25,Yes,Good,Under-7,M
2,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,10,7,0,30,No,Bad,Above-7,L
3,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,30,25,5,35,No,Bad,Above-7,L
4,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,40,50,12,50,No,Bad,Above-7,M


## 3. Create Pass/Fail Target Variable


In [5]:
# Create pass/fail from Class
# L = Low = Fail (0), M/H = Medium/High = Pass (1)
df_xapi['pass_fail'] = (df_xapi['Class'].isin(['M', 'H'])).astype(int)
df_xapi['pass_fail_label'] = df_xapi['pass_fail'].map({1: 'Pass', 0: 'Fail'})

print("xAPI Pass/Fail distribution:")
print(df_xapi['pass_fail_label'].value_counts())
print(f"\nPass rate: {df_xapi['pass_fail'].mean():.2%}")


xAPI Pass/Fail distribution:
pass_fail_label
Pass    353
Fail    127
Name: count, dtype: int64

Pass rate: 73.54%


## 4. Feature Engineering


In [6]:
# Create a copy for processing
df_xapi_processed = df_xapi.copy()

# 1. Gender: standardize values
if df_xapi_processed['gender'].dtype == 'object':
    df_xapi_processed['gender'] = df_xapi_processed['gender'].str.upper().str[0]  # M/F

# 2. Absences: StudentAbsenceDays (categorical) -> numeric
# Under-7 -> 3.5 (midpoint), Above-7 -> 10 (estimate)
df_xapi_processed['absences_numeric'] = df_xapi_processed['StudentAbsenceDays'].map({
    'Under-7': 3.5,
    'Above-7': 10.0
})

# 3. Engagement Score: composite from behavioral metrics
# Normalize each metric first, then combine
behavioral_cols = ['raisedhands', 'VisitedResources', 'AnnouncementsView', 'Discussion']

# Check if columns exist
for col in behavioral_cols:
    if col not in df_xapi_processed.columns:
        print(f"Warning: Column {col} not found. Available columns: {df_xapi_processed.columns.tolist()}")

# Normalize behavioral metrics to 0-1 scale
for col in behavioral_cols:
    if col in df_xapi_processed.columns:
        col_min = df_xapi_processed[col].min()
        col_max = df_xapi_processed[col].max()
        if col_max > col_min:
            df_xapi_processed[f"{col}_normalized"] = (df_xapi_processed[col] - col_min) / (col_max - col_min)
        else:
            df_xapi_processed[f"{col}_normalized"] = 0.0

# Create engagement score: weighted combination
if all(f"{col}_normalized" in df_xapi_processed.columns for col in behavioral_cols):
    df_xapi_processed['engagement_score'] = (
        df_xapi_processed['raisedhands_normalized'] * 0.3 +
        df_xapi_processed['VisitedResources_normalized'] * 0.3 +
        df_xapi_processed['AnnouncementsView_normalized'] * 0.2 +
        df_xapi_processed['Discussion_normalized'] * 0.2
    )
else:
    print("Warning: Could not create engagement_score. Using available metrics.")
    # Fallback: use average of available normalized metrics
    available_norm = [f"{col}_normalized" for col in behavioral_cols if f"{col}_normalized" in df_xapi_processed.columns]
    if available_norm:
        df_xapi_processed['engagement_score'] = df_xapi_processed[available_norm].mean(axis=1)
    else:
        df_xapi_processed['engagement_score'] = 0.0

# 4. Parental Support: derived from surveys
df_xapi_processed['parental_support'] = (
    (df_xapi_processed['ParentAnsweringSurvey'] == 'Yes') &
    (df_xapi_processed['ParentschoolSatisfaction'] == 'Good')
).astype(int)

# 5. Academic Level: derive from GradeID/StageID
# Map to numeric scale (higher grade/stage = higher academic level)
if 'GradeID' in df_xapi_processed.columns:
    # Create numeric mapping for GradeID
    grade_mapping = {}
    unique_grades = sorted(df_xapi_processed['GradeID'].unique())
    for idx, grade in enumerate(unique_grades):
        grade_mapping[grade] = idx + 1
    df_xapi_processed['academic_level'] = df_xapi_processed['GradeID'].map(grade_mapping)
elif 'StageID' in df_xapi_processed.columns:
    # Use StageID as fallback
    stage_mapping = {}
    unique_stages = sorted(df_xapi_processed['StageID'].unique())
    for idx, stage in enumerate(unique_stages):
        stage_mapping[stage] = idx + 1
    df_xapi_processed['academic_level'] = df_xapi_processed['StageID'].map(stage_mapping)
else:
    # Default: set to 1 if no grade/stage info
    df_xapi_processed['academic_level'] = 1

print("xAPI dataset processed.")
print(f"Shape: {df_xapi_processed.shape}")


xAPI dataset processed.
Shape: (480, 27)


## 5. Prepare Features for Processing


In [7]:
# Extract features for processing
df_xapi_features = pd.DataFrame()
df_xapi_features['gender'] = df_xapi_processed['gender']
df_xapi_features['absences'] = df_xapi_processed['absences_numeric']
df_xapi_features['engagement'] = df_xapi_processed['engagement_score']
df_xapi_features['parental_support'] = df_xapi_processed['parental_support']
df_xapi_features['academic_level'] = df_xapi_processed['academic_level']
df_xapi_features['pass_fail'] = df_xapi_processed['pass_fail']
df_xapi_features['pass_fail_label'] = df_xapi_processed['pass_fail_label']

print(f"Features extracted. Shape: {df_xapi_features.shape}")
print(f"\nColumns: {df_xapi_features.columns.tolist()}")


Features extracted. Shape: (480, 7)

Columns: ['gender', 'absences', 'engagement', 'parental_support', 'academic_level', 'pass_fail', 'pass_fail_label']


## 6. Encode Categorical Features


In [8]:
# Encode gender (M/F -> 0/1)
gender_encoder = LabelEncoder()
df_xapi_features['gender_encoded'] = gender_encoder.fit_transform(df_xapi_features['gender'])

print("Gender encoding:")
print(gender_encoder.classes_)
print(f"\nGender distribution:")
print(df_xapi_features['gender'].value_counts())


Gender encoding:
['F' 'M']

Gender distribution:
gender
M    305
F    175
Name: count, dtype: int64


## 7. Normalize Numeric Features


In [9]:
# Features to normalize
numeric_features = ['absences', 'engagement', 'academic_level']

# Fit scalers on the data
scalers = {}
for feature in numeric_features:
    scaler = MinMaxScaler()  # Normalize to 0-1 range
    scaler.fit(df_xapi_features[[feature]])
    scalers[feature] = scaler

# Transform features
for feature in numeric_features:
    df_xapi_features[f"{feature}_normalized"] = scalers[feature].transform(df_xapi_features[[feature]])

print("Features normalized successfully.")
print("\nNormalized feature ranges:")
for feature in numeric_features:
    print(f"  {feature}_normalized: [{df_xapi_features[f'{feature}_normalized'].min():.3f}, {df_xapi_features[f'{feature}_normalized'].max():.3f}]")


Features normalized successfully.

Normalized feature ranges:
  absences_normalized: [0.000, 1.000]
  engagement_normalized: [0.000, 1.000]
  academic_level_normalized: [0.000, 1.000]


## 8. Create Final Prepared Dataset


In [10]:
# Create final feature set (all normalized/encoded)
feature_columns = [
    'gender_encoded',
    'absences_normalized',
    'engagement_normalized',
    'parental_support',  # Already binary 0/1
    'academic_level_normalized'
]

# Create final dataset
df_xapi_final = pd.DataFrame()
for col in feature_columns:
    df_xapi_final[col] = df_xapi_features[col]
df_xapi_final['pass_fail'] = df_xapi_features['pass_fail']
df_xapi_final['pass_fail_label'] = df_xapi_features['pass_fail_label']

print("Final prepared dataset:")
print(f"Shape: {df_xapi_final.shape}")
print(f"Columns: {df_xapi_final.columns.tolist()}")

# Check for missing values
print(f"\nMissing values: {df_xapi_final.isnull().sum().sum()}")

# Display sample
print("\nSample (first 5 rows):")
df_xapi_final.head()


Final prepared dataset:
Shape: (480, 7)
Columns: ['gender_encoded', 'absences_normalized', 'engagement_normalized', 'parental_support', 'academic_level_normalized', 'pass_fail', 'pass_fail_label']

Missing values: 0

Sample (first 5 rows):


,gender_encoded,absences_normalized,engagement_normalized,parental_support,academic_level_normalized,pass_fail,pass_fail_label
0,1,0.0,0.139600,1,0.111111,1,Pass
1,1,0.0,0.181801,1,0.111111,1,Pass
2,1,1.0,0.111784,0,0.111111,0,Fail
3,1,1.0,0.256459,0,0.111111,0,Fail
4,1,1.0,0.417967,0,0.111111,1,Pass


## 9. Summary Statistics


In [11]:
print("=" * 60)
print("DATA PREPARATION SUMMARY")
print("=" * 60)

print(f"\nDataset size: {len(df_xapi_final)} samples")
print(f"Final feature set: {len(feature_columns)} features")
print(f"Final dataset shape: {df_xapi_final.shape}")

print("\nFeature Set:")
for i, feature in enumerate(feature_columns, 1):
    print(f"  {i}. {feature}")

print("\nTarget Distribution:")
print(df_xapi_final['pass_fail_label'].value_counts())
print(f"  Pass rate: {df_xapi_final['pass_fail'].mean():.2%}")

print("\nFeature Statistics:")
print(df_xapi_final[feature_columns].describe())

print("\n" + "=" * 60)
print("✓ Data preprocessing complete!")
print("=" * 60)


DATA PREPARATION SUMMARY

Dataset size: 480 samples
Final feature set: 5 features
Final dataset shape: (480, 7)

Feature Set:
  1. gender_encoded
  2. absences_normalized
  3. engagement_normalized
  4. parental_support
  5. academic_level_normalized

Target Distribution:
pass_fail_label
Pass    353
Fail    127
Name: count, dtype: int64
  Pass rate: 73.54%

Feature Statistics:
       gender_encoded  absences_normalized  engagement_normalized  \
count      480.000000           480.000000             480.000000   
mean         0.635417             0.397917               0.497353   
std          0.481815             0.489979               0.261018   
min          0.000000             0.000000               0.000000   
25%          0.000000             0.000000               0.264547   
50%          1.000000             0.000000               0.515873   
75%          1.000000             1.000000               0.729200   
max          1.000000             1.000000               1.000000   

## 10. Save Final Prepared Dataset


In [12]:
# Create output directory if it doesn't exist
# Save in final_code/datasets folder relative to the notebook location
output_dir = '../datasets'
os.makedirs(output_dir, exist_ok=True)

# Save the full prepared dataset
output_file = os.path.join(output_dir, 'xapi_pass_fail_prepared.csv')
df_xapi_final.to_csv(output_file, index=False)
print(f"✓ Saved full prepared dataset to: {output_file}")
print(f"  Shape: {df_xapi_final.shape}")
print(f"  Size: {os.path.getsize(output_file) / (1024*1024):.2f} MB")
print(f"\n✓ Dataset saved successfully!")


✓ Saved full prepared dataset to: ../datasets/xapi_pass_fail_prepared.csv
  Shape: (480, 7)
  Size: 0.02 MB

✓ Dataset saved successfully!
